In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd
from sklearn.linear_model import TheilSenRegressor
from sklearn.linear_model import LinearRegression

from matplotlib import gridspec
import scipy.stats as stats

from scipy.stats import kstest, cramervonmises
# import tensorflow as tf
# import tensorflow_probability as tfp
import pykrige.kriging_tools as kt
from pykrige.ok import OrdinaryKriging
import time

from ipcc_colormap import *
from utils import *


import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Myriad Pro'
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 600

coastline = gpd.read_file('coastlines-split-SGregion/lines.shp')
mask = np.loadtxt('mask.txt')

ipcc_blue = (112.0/255, 160.0/255, 205.0/255, 1.0)
ipcc_orange = (196.0/255, 121.0/255, 0.0/255, 1.0)

tmp_cmap = ipcc_cmap()
tmp_cmap.read_rgb_data_from_excel()
;

''

In [2]:
# DATA PREPARATION
# the station-based rainfall data is already organized in a N by P matrix
rain_obs = np.loadtxt('data/sta_monthly.csv')

# the simulated rainfall data is reshaped to a 2d matrix of size N by (W x L)
rain_sim_flatten = np.loadtxt('data/wrf_monthly.csv')
rain_sim = rain_sim_flatten.reshape(rain_sim_flatten.shape[0], 120, 160)

# sim_sel constains a P by 4 matrix
# the first two columns are the row and column indices of grids corresponding to the stations
# the last two columns are the lons and lats of grids corresponding to the stations
sim_sel = np.loadtxt('data/wrf_loc.csv')
sim_idx = sim_sel[:, :2].astype(int)

# select simulated rainfall at stations
wrf_sta = np.array([rain_sim[:, i, j] for (i, j) in sim_idx]).T

# lats and lons of all grids are stored in a 2d matrix of 2 by (W x L)
# the first row is lons and the second row is lats
longlat = np.loadtxt('data/lonlat.txt')
lons = longlat[0, :].reshape(120, 160)
lats = longlat[1, :].reshape(120, 160)

# read station lons (3rd column) and lats (4th column)
sta_loc = np.genfromtxt('data/sta_lookup_new.csv', delimiter=',')[:, 2:]

# Configuration constants
N_STATIONS = sim_idx.shape[0]  # 14 stations
N_MONTHS = 12
N_YEARS = 40
TOTAL_MONTHS = N_MONTHS * N_YEARS  # 480
FIGURE_DIR = 'figures'

In [3]:
def run_two_stage_validation(rain_obs, wrf_sta, sta_loc, bias_correct=False):
    """
    Runs leave-one-out cross validation for two-stage GP interpolation method.
    
    Args:
        rain_obs: Observed rainfall data (N x P matrix)
        wrf_sta: WRF simulated rainfall at station locations (N x P matrix)  
        sta_loc: Station coordinates (P x 2 matrix)
        bias_correct: Whether to apply bias correction to WRF data
        
    Returns:
        dict containing KGE scores, predictions, and KS statistics
    """
    # Apply bias correction if requested
    wrf_data = wrf_sta.copy()
    if bias_correct:
        wrf_data = np.zeros(wrf_sta.shape)
        for i in range(N_MONTHS):
            for j in range(N_STATIONS): 
                sorted_sim = np.sort(wrf_sta[i::12, j])
                sorted_obs = np.sort(rain_obs[i::12, j])
                lm = stats.linregress(sorted_sim, sorted_obs)
                wrf_data[i::12, j] = lm.slope * wrf_sta[i::12, j] + lm.intercept
    
    # Initialize result arrays
    kge_raw_wrf = np.zeros((N_MONTHS, N_STATIONS))
    kge_gp_stage1 = np.zeros((N_MONTHS, N_STATIONS))
    kge_gp_stage2 = np.zeros((N_MONTHS, N_STATIONS))
    ks_statistic = np.zeros((N_MONTHS, N_STATIONS))
    
    predictions_stage1 = np.zeros((TOTAL_MONTHS, N_STATIONS))
    predictions_stage2 = np.zeros((TOTAL_MONTHS, N_STATIONS))
    
    for month in range(N_MONTHS):
        # Extract data for this calendar month
        obs_month = rain_obs[month::12, :]
        sim_month = wrf_data[month::12, :]
        start_time = time.time()
        
        for station in range(N_STATIONS):
            # Leave out one station for evaluation
            mask = np.ones(N_STATIONS, dtype=bool)
            mask[station] = False
            
            train_obs = obs_month[:, mask]
            train_sim = sim_month[:, mask]
            train_loc = sta_loc[mask, :]
            
            # Target station data
            target_sim = sim_month[:, station][:, None]
            target_obs = obs_month[:, station][:, None]
            
            # Stage 1: GP interpolation using WRF covariances
            gp = gp_interpolator(P=N_STATIONS - 1)
            gp.read_rainfall(obs=train_obs, sim=train_sim)
            gp.sn_converge()
            
            stage1_pred, _ = gp.predict(target_sim)
            stage1_train, _ = gp.predict(train_sim)
            
            # Stage 2: Kriging of residuals
            residuals = train_obs - stage1_train.T
            stage2_pred = np.zeros(stage1_pred.shape)
            
            for time_idx in range(residuals.shape[0]):
                residual_data = np.concatenate((train_loc, residuals[time_idx, :][:, None]), axis=1)
                OK = OrdinaryKriging(
                    residual_data[:, 0], residual_data[:, 1], residual_data[:, 2],
                    variogram_model='gaussian'
                )
                kriged_residual, _ = OK.execute('points', sta_loc[station, 0], sta_loc[station, 1])
                stage2_pred[time_idx] = stage1_pred[time_idx] + kriged_residual
            
            # Ensure non-negative predictions
            stage1_pred[stage1_pred < 0] = 0
            stage2_pred[stage2_pred < 0] = 0
            
            # Calculate performance metrics
            kge_raw_wrf[month, station] = kge(target_obs.squeeze(), target_sim.squeeze())
            kge_gp_stage1[month, station] = kge(target_obs.squeeze(), stage1_pred.squeeze())
            kge_gp_stage2[month, station] = kge(target_obs.squeeze(), stage2_pred.squeeze())
            
            # KS test for prior adequacy (only for non-bias-corrected case)
            if not bias_correct:
                z_values = stats.norm.cdf(target_obs.squeeze(), 
                                        loc=np.mean(target_sim), 
                                        scale=np.std(target_sim, ddof=1))
                ks_result = kstest(np.sort(z_values), 'uniform')
                ks_statistic[month, station] = ks_result.statistic
            
            # Store predictions
            predictions_stage1[month::12, station] = stage1_pred.squeeze()
            predictions_stage2[month::12, station] = stage2_pred.squeeze()
        
        print('Month %d: %.2f seconds' % (month + 1, time.time() - start_time))
    
    return {
        'kge_raw_wrf': kge_raw_wrf,
        'kge_gp_stage1': kge_gp_stage1, 
        'kge_gp_stage2': kge_gp_stage2,
        'ks_statistic': ks_statistic,
        'predictions_stage1': predictions_stage1,
        'predictions_stage2': predictions_stage2
    }

In [4]:
# Run two-stage validation WITHOUT bias correction
print("Running validation with raw WRF simulations...")
results_raw = run_two_stage_validation(rain_obs, wrf_sta, sta_loc, bias_correct=False)

Running validation with raw WRF simulations...
1
[np.float64(74.61567456867073), np.float64(77.97939822256174), np.float64(84.11371284262262), np.float64(79.48107453570567), np.float64(105.15146129328035), np.float64(113.34049417080203), np.float64(77.16395443081977), np.float64(66.49538592335841), np.float64(57.550357770807416), np.float64(75.17497559951624), np.float64(70.58845803101214), np.float64(55.549648713941295), np.float64(63.92324543597488)]
2
[np.float64(47.01537910746829), np.float64(43.77248039801315), np.float64(44.92787269283927), np.float64(86.1518781049005), np.float64(95.1002756764305), np.float64(79.79018259651272), np.float64(45.203204244881), np.float64(50.02783751182464), np.float64(40.989216714654816), np.float64(43.83256183445831), np.float64(55.73674824764871), np.float64(44.078445533428905), np.float64(56.27489623068842)]
3
[np.float64(46.87202237638742), np.float64(42.07126457034244), np.float64(45.88478937229629), np.float64(90.53361263053729), np.float64(9

In [5]:
kge_kriging = np.loadtxt('kriging_kge.csv')
# the KGE boxplot
fig, ax = plt.subplots(1, 1, figsize = (6, 3))
# plot the kge0 boxplot at positions of x with filler color of red
bp1 = ax.boxplot(results_raw['kge_raw_wrf'].T, positions = np.arange(12) - 0.1, widths = 0.15,
           patch_artist = True, boxprops=dict(facecolor=ipcc_orange), medianprops=dict(color="black"))
bp2 = ax.boxplot(results_raw['kge_gp_stage2'].T, positions = np.arange(12) + 0.1, widths = 0.15,
           patch_artist = True, boxprops=dict(facecolor=ipcc_blue), medianprops=dict(color="black"))

# ax.axline((0, 0), (11, 0), color = 'black', linestyle = '--')
ax.set_xticks(np.arange(12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.set_ylabel('KGE')
ax.set_ylim([-0.5, 1])
ax.set_yticks([-0.5, 0, 0.5, 1])
# set minor ticks on y axis with an interval of 0.25
ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.25))
ax.grid(which = 'both', axis = 'y', linestyle = '--')
# Place the legend horizontally above the plot
handles = [bp1["boxes"][0], bp2["boxes"][0]]
labels = ['Raw', 'Posterior']
ax.legend(handles, labels, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.15), fontsize=10, frameon=False)

np.sum((results_raw['kge_gp_stage2'] >= kge_kriging), axis = 1) / N_STATIONS
fig.savefig(f'{FIGURE_DIR}/kge_boxplot_for_presentation.png', dpi = 600, bbox_inches = 'tight')

In [6]:
kge_kriging = np.loadtxt('kriging_kge.csv')
# the KGE boxplot
fig, ax = plt.subplots(1, 1, figsize = (6, 3))
# plot the kge0 boxplot at positions of x with filler color of red
bp1 = ax.boxplot(results_raw['kge_gp_stage1'].T, positions = np.arange(12) - 0.1, widths = 0.15,
           patch_artist = True, boxprops=dict(facecolor=ipcc_orange), medianprops=dict(color="black"))
bp2 = ax.boxplot(results_raw['kge_gp_stage2'].T, positions = np.arange(12) + 0.1, widths = 0.15,
           patch_artist = True, boxprops=dict(facecolor=ipcc_blue), medianprops=dict(color="black"))
bp3 = ax.scatter(np.arange(12) + 0.1, np.median(kge_kriging.T, axis = 0), s = 15, marker = '^', color = 'limegreen', zorder = 10)
bp4 = ax.scatter(np.arange(12) - 0.1, np.median(results_raw['kge_raw_wrf'].T, axis = 0), s = 15, marker = 'D', color = 'black', zorder = 10)
ax.axhline(y=-0.41, color='black', linestyle='--', linewidth=0.8)

ax.set_xticks(np.arange(12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.set_ylabel('KGE')
ax.set_ylim([-0.5, 1])
ax.set_yticks([-0.5, -0.41, 0, 0.5, 1])
ax.set_yticklabels(['-0.5', '-0.41', '0', '0.5', '1'])
# set minor ticks on y axis with an interval of 0.25
ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.25))
ax.grid(which = 'both', axis = 'y', linestyle = '--')
# hide the grey grid line that falls on the -0.41 tick
for line in ax.get_ygridlines():
    if abs(line.get_ydata()[0] - (-0.41)) < 1e-6:
        line.set_visible(False)
# Place the legend horizontally above the plot
handles = [bp1["boxes"][0], bp2["boxes"][0], bp3, bp4]
labels = ['Stage 1', 'Stage 1 + 2', 'Kriging', 'Raw']
ax.legend(handles, labels, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.15), fontsize=10, frameon=False)

np.sum((results_raw['kge_gp_stage2'] >= kge_kriging), axis = 1) / N_STATIONS
fig.savefig(f'{FIGURE_DIR}/kge_boxplot.png', dpi = 600, bbox_inches = 'tight')

In [7]:
fig, ax = plt.subplots(1, 1, figsize = (6, 3))
bp = ax.boxplot(results_raw['ks_statistic'].T, positions = np.arange(12), widths = 0.3)
ax.set_xticks(np.arange(12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.set_ylabel('K-S Statistic')
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1])
# set minor ticks on y axis with an interval of 0.25
ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.2))
ax.grid(which = 'both', axis = 'y', linestyle = '--')
fig.savefig(f'{FIGURE_DIR}/ks_statistic.png', dpi = 600, bbox_inches = 'tight')

In [8]:
# Run two-stage validation WITH bias correction
print("Running validation with bias-corrected WRF simulations...")
results_bc = run_two_stage_validation(rain_obs, wrf_sta, sta_loc, bias_correct=True)

Running validation with bias-corrected WRF simulations...
1
[np.float64(69.17814230494099), np.float64(69.95173642345854), np.float64(81.94770717115027), np.float64(83.04437236950886), np.float64(93.38733205331937), np.float64(60.73345901178981), np.float64(57.914134240637175), np.float64(68.8850460328546), np.float64(64.78580272024446), np.float64(55.1385958974542), np.float64(71.29476808757116), np.float64(56.14065770250701), np.float64(67.54701130163724)]
2
[np.float64(42.64181865825311), np.float64(39.7087598530321), np.float64(46.31620715056242), np.float64(56.608203134283585), np.float64(75.98857248166192), np.float64(40.77033185639099), np.float64(40.301355179630534), np.float64(51.465310697696786), np.float64(34.58526165999533), np.float64(31.176976964731878), np.float64(49.412225910290736), np.float64(41.20869114773594), np.float64(47.30161065794463)]
3
[np.float64(43.329421733696954), np.float64(38.554343228971206), np.float64(45.082377834361964), np.float64(58.15520451751655

In [9]:
# for plotting bias corrected KGE (approximate of upper bound)
kge_kriging = np.loadtxt('kriging_kge.csv')
# the KGE boxplot
fig, ax = plt.subplots(1, 1, figsize = (6, 3))
# plot the kge0 boxplot at positions of x with filler color of red
bp1 = ax.boxplot(results_bc['kge_gp_stage1'].T, positions = np.arange(12) - 0.1, widths = 0.15,
           patch_artist = True, boxprops=dict(facecolor=ipcc_orange), medianprops=dict(color="black"))
bp2 = ax.boxplot(results_bc['kge_gp_stage2'].T, positions = np.arange(12) + 0.1, widths = 0.15,
           patch_artist = True, boxprops=dict(facecolor=ipcc_blue), medianprops=dict(color="black"))
bp3 = ax.scatter(np.arange(12) + 0.1, np.median(kge_kriging.T, axis = 0), s = 15, marker = '^', color = 'limegreen', zorder = 10)
bp4 = ax.scatter(np.arange(12) - 0.1, np.median(results_bc['kge_raw_wrf'].T, axis = 0), s = 15, marker = 'D', color = 'black', zorder = 10)
ax.axhline(y=-0.41, color='black', linestyle='--', linewidth=0.8)

ax.set_xticks(np.arange(12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.set_ylabel('KGE')
ax.set_ylim([-0.5, 1])
ax.set_yticks([-0.5, -0.41, 0, 0.5, 1])
ax.set_yticklabels(['-0.5', '-0.41', '0', '0.5', '1'])
# set minor ticks on y axis with an interval of 0.25
ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.25))
ax.grid(which = 'both', axis = 'y', linestyle = '--')
# hide the grey grid line that falls on the -0.41 tick
for line in ax.get_ygridlines():
    if abs(line.get_ydata()[0] - (-0.41)) < 1e-6:
        line.set_visible(False)
# Place the legend horizontally above the plot
handles = [bp1["boxes"][0], bp2["boxes"][0], bp3, bp4]
labels = ['Stage 1', 'Stage 1 + 2', 'Kriging', 'Raw']
ax.legend(handles, labels, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.15), fontsize=10, frameon=False)

fig.savefig(f'{FIGURE_DIR}/kge_boxplot_bc.png', dpi = 600, bbox_inches = 'tight')
np.sum((results_bc['kge_gp_stage2'] >= kge_kriging), axis = 1) / N_STATIONS

array([0.57142857, 0.92857143, 0.71428571, 0.78571429, 0.78571429,
       0.64285714, 0.71428571, 0.78571429, 0.71428571, 0.85714286,
       0.71428571, 0.64285714])

In [10]:
fig, ax = plt.subplots(1, 3, figsize = (9, 3.4),sharey=True)

tmp_x = results_raw['ks_statistic'].flatten()
tmp_y = results_bc['kge_gp_stage2'].flatten() - results_raw['kge_gp_stage1'].flatten()

ax[0].scatter(tmp_x, tmp_y, s = 15, marker='o')
ax[0].set_ylim([-0.2, 0.6])
ax[0].set_ylabel('KGE Difference')

xmin = np.min(tmp_x)
xmax = np.max(tmp_x)
xx_v = np.linspace(xmin, xmax, 100)
# delete where tmp_y < -0.2
# drop two station-month outliers
# cus they are stretching the whole y-axis scale too much
outlier = np.where(tmp_y < -0.2)
tmp_x = np.delete(tmp_x, outlier)
tmp_y = np.delete(tmp_y, outlier)

expo_ = 1.8 # thru trial and error, 1.8 is the best
lm = LinearRegression(fit_intercept = False).fit(tmp_x[:, None]**expo_, tmp_y)
yy_v = lm.predict(xx_v[:, None]**expo_)
print(lm.coef_)
ax[0].plot(xx_v, yy_v, color = 'black', linestyle = '--')

RSS = np.sum((tmp_y - lm.predict(tmp_x[:, None]**expo_))**2)
TSS = np.sum((tmp_y - np.mean(tmp_y))**2)
rsq = 1 - RSS / TSS
print([rsq, np.sqrt(rsq)])

a = lm.coef_[0]
formula_text = r"$Y = {:.2f} \cdot X^{{{:.1f}}}$".format(a, expo_)
ax[0].text(0.23, 0.5, formula_text, fontsize=10, ha='center', va='center')
ax[0].set_title('(a) $R^2=$ {:.2f}'.format(rsq), fontsize = 10)

#----------------------------------------------------------------------------------
# Second subplot: kge1_bc vs kge1 difference
tmp_x = results_raw['ks_statistic'].flatten()
tmp_y = results_bc['kge_gp_stage1'].flatten() - results_raw['kge_gp_stage1'].flatten()

ax[1].scatter(tmp_x, tmp_y, s = 15, marker='o')
ax[1].set_ylim([-0.2, 0.6])
ax[1].set_xlabel('KS Statistic')

xmin = np.min(tmp_x)
xmax = np.max(tmp_x)
xx_v = np.linspace(xmin, xmax, 100)
# drop outliers
outlier = np.where(tmp_y < -0.2)
tmp_x = np.delete(tmp_x, outlier)
tmp_y = np.delete(tmp_y, outlier)

expo_ = 2.4 # thru trial and error, 2.4 is the best
lm = LinearRegression(fit_intercept = False).fit(tmp_x[:, None]**expo_, tmp_y)
yy_v = lm.predict(xx_v[:, None]**expo_)
print(lm.coef_)
ax[1].plot(xx_v, yy_v, color = 'black', linestyle = '--')

RSS = np.sum((tmp_y - lm.predict(tmp_x[:, None]**expo_))**2)
TSS = np.sum((tmp_y - np.mean(tmp_y))**2)
rsq = 1 - RSS / TSS
print([rsq, np.sqrt(rsq)])

a = lm.coef_[0]
formula_text = r"$Y = {:.2f} \cdot X^{{{:.1f}}}$".format(a, expo_)
ax[1].text(0.23, 0.5, formula_text, fontsize=10, ha='center', va='center')
ax[1].set_title('(b) $R^2=$ {:.2f}'.format(rsq), fontsize = 10)

#----------------------------------------------------------------------------------
# Third subplot: kge2 vs kge1 difference
tmp_x = results_raw['ks_statistic'].flatten()
tmp_y = results_raw['kge_gp_stage2'].flatten() - results_raw['kge_gp_stage1'].flatten()

ax[2].scatter(tmp_x, tmp_y, s = 15, marker='o')
ax[2].set_ylim([-0.2, 0.6])

xmin = np.min(tmp_x)
xmax = np.max(tmp_x)
xx_v = np.linspace(xmin, xmax, 100)
# drop outliers
outlier = np.where(tmp_y < -0.2)
tmp_x = np.delete(tmp_x, outlier)
tmp_y = np.delete(tmp_y, outlier)

expo_ = 2.5 # thru trial and error, 2.5 is the best
lm = LinearRegression(fit_intercept = False).fit(tmp_x[:, None]**expo_, tmp_y)
yy_v = lm.predict(xx_v[:, None]**expo_)
ax[2].plot(xx_v, yy_v, color = 'black', linestyle = '--')
print(lm.coef_)

RSS = np.sum((tmp_y - lm.predict(tmp_x[:, None]**expo_))**2)
TSS = np.sum((tmp_y - np.mean(tmp_y))**2)
rsq = 1 - RSS / TSS
print([rsq, np.sqrt(rsq)])

a = lm.coef_[0]
formula_text = r"$Y = {:.2f} \cdot X^{{{:.1f}}}$".format(a, expo_)
ax[2].text(0.23, 0.5, formula_text, fontsize=10, ha='center', va='center')
ax[2].set_title('(c) $R^2=$ {:.2f}'.format(rsq), fontsize = 10)

plt.tight_layout()
fig.savefig(f'{FIGURE_DIR}/KGE_diff_vs_KS.png', dpi = 600, bbox_inches = 'tight')

[0.59433003]
[np.float64(0.3283978182560777), np.float64(0.573060047687917)]
[0.67518551]
[np.float64(0.41109997570954837), np.float64(0.6411707851341547)]
[0.53310887]
[np.float64(0.4085503368887621), np.float64(0.6391794246444124)]


In [11]:
# the scatter plots of raw vs posterior mean
sta_name = pd.read_csv('data/sta_lookup_new.csv', delimiter=',', header=None)[0]
scatter_plot_path = f'{FIGURE_DIR}/scatter_plots/'
subplot_labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n']

for month in range(N_MONTHS):
    obs_month = rain_obs[month::12, :]
    sim_month = wrf_sta[month::12, :]
    fig, ax = plt.subplots(4, 4, figsize=(12, 12), sharex=True, sharey=True)
    
    for station in range(N_STATIONS):
        row_idx, col_idx = divmod(station, 4)  # Calculate row, column index
        ax[row_idx, col_idx].scatter(obs_month[:, station], sim_month[:, station], 
                                   color='dimgrey', marker='D', s=50, alpha=0.4, 
                                   edgecolor='none', label='Raw')
        ax[row_idx, col_idx].scatter(obs_month[:, station], results_raw['predictions_stage1'][month::12, station], 
                                   color=ipcc_orange, marker='o', s=50, alpha=0.8, 
                                   edgecolor='black', label='Stage 1')
        ax[row_idx, col_idx].scatter(obs_month[:, station], results_raw['predictions_stage2'][month::12, station], 
                                   color=ipcc_blue, marker='^', s=50, alpha=0.8, 
                                   edgecolor='black', label='Stage 1 + 2')
        ax[row_idx, col_idx].axline([0, 0], [1, 1], color='black', linestyle='--')
        ax[row_idx, col_idx].set_xlim(left=0)  # Set x-axis to start from 0
        ax[row_idx, col_idx].set_ylim(bottom=0)  # Set y-axis to start from 0
        
        # Calculate KGE scores for this station-month
        kge_raw = kge(obs_month[:, station], sim_month[:, station])
        kge_stage1 = kge(obs_month[:, station], results_raw['predictions_stage1'][month::12, station])
        kge_stage2 = kge(obs_month[:, station], results_raw['predictions_stage2'][month::12, station])
        
        ax[row_idx, col_idx].set_title(f'({subplot_labels[station]}) KGE = {kge_stage1:.2f}, {kge_stage2:.2f} ({kge_raw:.2f})')
        # Add station names inside subplots as legends
        ax[row_idx, col_idx].text(0.05, 0.95, f'{sta_name[station]}', transform=ax[row_idx, col_idx].transAxes, 
                        fontsize=10, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.6))
    
    # Hide unused subplots
    for k in range(N_STATIONS, 16):
        ax[k // 4, k % 4].axis('off')
    
    ax[2, 2].tick_params(axis='x', which='both', labelbottom=True)  # Figure (k)
    ax[2, 3].tick_params(axis='x', which='both', labelbottom=True)  # Figure (l)
    
    # Place a single legend on the last row's leftmost subplot
    handles, labels = ax[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower left', bbox_to_anchor=(0.6, 0.1), fontsize = 14)

    # Add shared x and y axis labels
    fig.text(0.5, 0.01, 'Observed Rainfall [mm]', ha='center', va='center', fontsize='large')
    fig.text(0.01, 0.5, 'Estimated Rainfall [mm]', ha='center', va='center', rotation='vertical', fontsize='large')
    plt.subplots_adjust(left=0.12, right=0.9, top=0.9, bottom=0.12)  # Adjust the subplots to provide space for labels
    plt.tight_layout(pad=2)  # Increase padding to avoid overlap

    fig.savefig(f'{scatter_plot_path}scatter_{month+1}.png', dpi=600, bbox_inches='tight')
    plt.close(fig)